In [1]:
# Section 1: Setup Env

%pip install pyhealth
# %pip install scikit-learn matplotlib seaborn
%pip install pandas numpy torch scikit-learn



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [71]:
# ==========================================
# Step 1: Load MIMIC-III dataset with PyHealth
# ==========================================

from pyhealth.datasets import MIMIC3Dataset
import pandas as pd

DATA_PATH = "../MIMIC3/raw" # CHANGE THIS TO YOUR FOLDER PATH OF THE UNZIPPED FILES

mimic_dataset = MIMIC3Dataset(
    root=DATA_PATH,
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS", "LABEVENTS"],
    code_mapping = {"ICD9CM": "CCSCM", "ICD9PROC": "CCSPROC", "NDC": "ATC"},
    dev=True
)


chart_df = pd.read_csv(DATA_PATH + "/CHARTEVENTS.csv", usecols=["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUE"])
d_items = pd.read_csv(DATA_PATH + "/D_ITEMS.csv")
patients_dob_df = pd.read_csv(DATA_PATH + "/PATIENTS.csv", usecols=["SUBJECT_ID", "DOB"], engine="python")

/var/folders/2_/z9nw__lj1zb_qbwcy8742qhw0000gn/T/ipykernel_38629/3244824230.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  chart_df = pd.read_csv(DATA_PATH + "/CHARTEVENTS.csv", usecols=["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUE"])


In [ ]:
# Define a function to check if the patient is over 18
from datetime import datetime

def is_over_18(df, id_col, dob_col, target_id):
    # Locate the row matching the target ID
    dob_raw = df.loc[df[id_col] == target_id, dob_col]

    if dob_raw.empty:
        return False  # ID not found

    # Try parsing the DOB
    dob_parsed = pd.to_datetime(dob_raw.values[0], errors='coerce', infer_datetime_format=True)
    print(dob_parsed)
    if pd.isna(dob_parsed):
        return False  # Could not parse date

    # Calculate age
    today = pd.Timestamp.today()
    age_years = (today - dob_parsed).days // 365
    print("Patient Age: ", age_years)
    
    return age_years >= 18

1864-11-16 00:00:00
Patient Age:  160
True


In [74]:
# Look at the top matches

# Define vital sign types and associated keywords
vital_keywords = {
    "heartrate": ["heart rate"],
    "resprate": ["respiratory rate"],
    "sysbp": ["systolic"],
    "diabp": ["diastolic"],
    "meanbp": ["mean blood pressure", "arterial bp mean", "non invasive bp mean", "bp mean", "blood pressure mean",],
    "spo2": ["spo2", "oxygen saturation"],
    "temperature": ["temperature", "body temperature"]
}

# Function to search for ITEMIDs
def find_itemids(df, keywords):
    itemids = set()
    for kw in keywords:
        matches = df[df["LABEL"].str.contains(kw, case=False, na=False)]
        itemids.update(matches["ITEMID"].astype(str).tolist())
    return itemids

# Build the dictionary
vital_itemids = {
    vital: find_itemids(d_items, keywords)
    for vital, keywords in vital_keywords.items()
}

# Print the result
for vital, itemids in vital_itemids.items():
    print(f"{vital}: {sorted(itemids)}")


# Flatten ITEMIDs to a set
all_vital_itemids = set().union(*vital_itemids.values())

# Filter rows with those ITEMIDs
chart_df = chart_df[chart_df["ITEMID"].astype(str).isin(all_vital_itemids)]
# Convert CHARTTIME to datetime for grouping
chart_df["CHARTTIME"] = pd.to_datetime(chart_df["CHARTTIME"], errors='coerce')
chart_df["VALUE"] = pd.to_numeric(chart_df["VALUE"], errors='coerce')

# Drop NaNs
chart_df.dropna(subset=["VALUE"], inplace=True)

# Map ITEMIDs to vital names
itemid_to_vital = {itemid: vital for vital, ids in vital_itemids.items() for itemid in ids}
chart_df["vital_type"] = chart_df["ITEMID"].astype(str).map(itemid_to_vital)

# Aggregate: mean value per vital per visit
agg_vitals = chart_df.groupby(["SUBJECT_ID", "HADM_ID", "vital_type"])["VALUE"].mean().unstack("vital_type")
agg_vitals.reset_index(inplace=True)


heartrate: ['211', '220045', '220046', '220047', '3494']
resprate: ['220210', '224688', '224689', '224690', '618', '619']
sysbp: ['220050', '220059', '220179', '224167', '225309', '226850', '226852', '227243', '228152', '3313', '3315', '3317', '3319', '3321', '3323', '3325', '442', '455', '480', '482', '484', '492', '51', '6', '666', '6701', '7643']
diabp: ['153', '220051', '220060', '220180', '224643', '225310', '226851', '226853', '227242', '228151', '8364', '8368', '8440', '8441', '8444', '8445', '8446', '8448', '8502', '8503', '8504', '8505', '8506', '8507', '8508', '8555']
meanbp: ['220052', '220181', '224', '224322', '225312', '443', '456', '52', '5731', '6653', '6702']
spo2: ['226253', '228232', '5820', '646', '6719', '8554']
temperature: ['223761', '223762', '224027', '224642', '224674', '226329', '227054', '228242', '591', '597', '645', '676', '677', '678', '679', '8537']


In [ ]:
import numpy as np
from scipy.io import loadmat
from pyhealth.data import Patient, Visit

# Helper function to get vitals per patient per visit
def get_vitals_for_visit(patient_id, hadm_id):
    row = agg_vitals[(agg_vitals["SUBJECT_ID"] == int(patient_id)) & (agg_vitals["HADM_ID"] == int(hadm_id))]
    return row.iloc[0].to_dict() if not row.empty else {}

def aki_detection_fn(patient: Patient, time_window=7):
    """Processes a single patient for AKI detection.

    The goal is to detect Acute Kidney Injury (AKI) in patients based on clinical data.
    This function reads patient records, extracts relevant biomarkers, and creates
    epochs to train a classifier on AKI detection (binary classification task: AKI or no AKI).

    Args:
        record: a list of patient records, where each record is a dictionary containing:
            - 'patient_id': unique identifier for the patient
            - 'signal_file': path to the signal (clinical data) file
            - 'label_file': path to the label file indicating AKI diagnosis (ICD-10/SNOMED codes)
            - 'save_to_path': directory to save the output epochs
        epoch_sec: length of each epoch in seconds
        shift: step size for sliding window to create epochs

    Returns:
        samples: list of dictionaries containing:
            - 'patient_id': patient identifier
            - 'visit_id': visit or record identifier
            - 'record_id': unique identifier for the record
            - 'epoch_path': path to the saved epoch pickle file
            - 'label': 1 for AKI, 0 for non-AKI
    """


    # Define AKI-related diagnosis codes
    aki_icd9_codes = {"5845", "5846", "5847", "5848", "5849", "5844"}
    samples = []

    # ================ DOB CHECK NOT WORKING BECAUSE DATA IN MIMIC IS BAD =====================
    # Only add samples for patients over 18 years old
    # patientOver18 = is_over_18(patients_dob_df, id_col="SUBJECT_ID", dob_col="DOB", target_id=patient.patient_id)
    # if not patientOver18: return samples
    
    visitItems = list(patient.visits.items())
    for i, (visit_id, visit) in enumerate(visitItems):
        next_visit = visitItems[i + 1] if i < len(visitItems) - 1 else None
        (_, next_visit_obj) = next_visit if next_visit is not None else (None, None)
        
        if (visit.encounter_time - patient.birth_datetime).days // 365 < 18:
            continue

        conditions = visit.get_code_list("DIAGNOSES_ICD")
        procedures = visit.get_code_list("PROCEDURES_ICD")
        drugs = visit.get_code_list("PRESCRIPTIONS")
        # skip if no data
        # if len(conditions) == 0 or len(procedures) == 0 or len(drugs) == 0:
        #     continue

        # time delta
        delta = 0 if next_visit_obj is None else (next_visit_obj.encounter_time - visit.encounter_time).days # PAPER RECOMMENDING FROM 24-48 HOURS before detected AKI

        # If AKI shows up in next visit within prediction_window, label current visit
        future_conditions = set() if next_visit is None else set(next_visit_obj.get_code_list("DIAGNOSES_ICD"))
        aki_label = 1 if len(aki_icd9_codes & future_conditions) > 0 and delta <= time_window else 0

        vitals = get_vitals_for_visit(patient.patient_id, visit_id)
        # print(vitals)

        samples.append({
            "visit_id": visit_id,
            "patient_id": patient.patient_id,
            "conditions": [conditions],
            "procedures": [procedures],
            "drugs": [drugs],
            "label": aki_label,
            "gender": patient.gender,
            "vitals": str(vitals)
        })

    return samples

mimic_aki_processed_dataset = mimic_dataset.set_task(aki_detection_fn)

Generating samples for aki_detection_fn:  12%|█▏        | 119/1000 [00:00<00:01, 658.25it/s]

[]
[{'visit_id': '153952', 'patient_id': '100', 'conditions': [['237', '96', '115', '105', '106']], 'procedures': [['43', '52', '50', '48']], 'drugs': [[]], 'label': 0, 'gender': 'F', 'vitals': "{'SUBJECT_ID': 100.0, 'HADM_ID': 153952.0, 'diabp': 41.483870967741936, 'heartrate': 86.62921348314607, 'meanbp': 77.47253187410124, 'resprate': 15.587628865979381, 'spo2': 97.24444444444444, 'sysbp': 75.91612903225807, 'temperature': 67.83436160301095}"}]
[{'visit_id': '175533', 'patient_id': '101', 'conditions': [['131', '129', '122', '238', '107', '85', '55']], 'procedures': [['216', '54', '39']], 'drugs': [[]], 'label': 0, 'gender': 'M', 'vitals': "{'SUBJECT_ID': 101.0, 'HADM_ID': 175533.0, 'diabp': 57.682403433476395, 'heartrate': 75.70869565217392, 'meanbp': 82.52011511243623, 'resprate': 21.44176706827309, 'spo2': 98.57317073170732, 'sysbp': 133.4102564102564, 'temperature': 68.00222438176473}"}]
[]
[{'visit_id': '130744', 'patient_id': '103', 'conditions': [['42', '109', '19', '122', '9

Generating samples for aki_detection_fn:  32%|███▏      | 324/1000 [00:00<00:00, 896.15it/s]

[{'visit_id': '139826', 'patient_id': '1395', 'conditions': [['100', '101', '249', '238', '106', '49', '204', '98', '53']], 'procedures': [['45', '63', '49', '48', '47', '231', '225', '193']], 'drugs': [['C09AA01', 'C01CA07', 'B01AC16', 'B01AB01', 'B05XA01', 'C07AB02', 'N02AJ17', 'B01AC04', 'A12BA01', 'A03BA01', 'N02BA01', 'N02BE01', 'N06AX05', 'A02BC02', 'A06AA02', 'A04AA01', 'A02AB01', 'A02AB02', 'A02AA04', 'G04BX01', 'C10AA05', 'A02AA02', 'A06AD02', 'A12CC10', 'C03CA01', 'B01AA03', 'N05CF02', 'M01AE02', 'M01AE18', 'C09AA03']], 'label': 0, 'gender': 'M', 'vitals': "{'SUBJECT_ID': 1395.0, 'HADM_ID': 139826.0, 'diabp': 53.559633027522935, 'heartrate': 102.45454545454545, 'meanbp': 97.87581956152822, 'resprate': 22.446153846153845, 'spo2': 98.703125, 'sysbp': 66.05673758865248, 'temperature': 67.9777785709926}"}]
[{'visit_id': '169346', 'patient_id': '1396', 'conditions': [['101', '238', '106', '159', '98', '53']], 'procedures': [['44', '50', '49', '47']], 'drugs': [[]], 'label': 0, 'ge

Generating samples for aki_detection_fn:  52%|█████▏    | 515/1000 [00:00<00:00, 890.29it/s]

...

[{'visit_id': '132048', 'patient_id': '593', 'conditions': [['233', '106', '95', '98', '2603']], 'procedures': [['216']], 'drugs': [[]], 'label': 0, 'gender': 'F', 'vitals': "{'SUBJECT_ID': 593.0, 'HADM_ID': 132048.0, 'diabp': 49.48, 'heartrate': 127.38461538461539, 'meanbp': 67.90666885375977, 'resprate': 17.548387096774192, 'spo2': 89.42857142857143, 'sysbp': 104.76, 'temperature': 70.72020860151811}"}]
[{'visit_id': '114388', 'patient_id': '594', 'conditions': [['234', '130', '109', '55', '131', '231', '229']], 'procedures': [['34', '89', '39', '204', '71', '193', '222', '54', '8', '216']], 'drugs': [['N05CD08', 'N01AH01', 'A02BA03', 'B01AB01', 'J01DB04', 'J01XD01', 'V06DC01', 'C01CA03', 'J01MA12', 'N05BA06', 'M03AC11', 'B05AA01', 'N01AX10', 'C01CA04', 'A02BC02', 'B05XA03', 'A12AA03', 'B05XA01', 'C03CA01', 'C07AB02', 'B01AB05', 'M01AH02', 'A02BC03', 'S01EC01', 'N06BA04', 'N02AJ17']], 'label': 0, 'gender': 'F', 'vitals': "{'SUBJECT_ID': 594.0, 'HADM_ID': 114388.0, 'diabp': 45.7508417

Generating samples for aki_detection_fn:  89%|████████▊ | 887/1000 [00:01<00:00, 589.56it/s]

[{'visit_id': '175906', 'patient_id': '718', 'conditions': [['233', '109', '99', '2607', '95']], 'procedures': [['216']], 'drugs': [['N02AA01', 'C02DD01', 'N01AX10', 'V06DC01']], 'label': 0, 'gender': 'M', 'vitals': "{'SUBJECT_ID': 718.0, 'HADM_ID': 175906.0, 'diabp': 54.666666666666664, 'heartrate': 77.4, 'meanbp': 74.66666666666667, 'resprate': 26.0, 'spo2': 75.75, 'sysbp': 114.66666666666667, 'temperature': 65.77779960632324}"}]
[{'visit_id': '180826', 'patient_id': '719', 'conditions': [['109', '122', '157']], 'procedures': [['9', '54']], 'drugs': [['C02DD01', 'N03AB02', 'C07AB02', 'V06DC01', 'B05XA03', 'J01MA12', 'A02BC02', 'B02BA01', 'A04AA01', 'C07AG01', 'A02AA02', 'A06AD02', 'A12CC10', 'A06AH04', 'V03AB15', 'A02BC03', 'R03AC02', 'N02BE01', 'R06AA02', 'N02AA01', 'N01AH01', 'A06AB02', 'A06AD10', 'A06AG02', 'A06AA02', 'A06AD65', 'J01DB04']], 'label': 0, 'gender': 'M', 'vitals': "{'SUBJECT_ID': 719.0, 'HADM_ID': 180826.0, 'diabp': 59.48453608247423, 'heartrate': 93.16955017301038, 

Generating samples for aki_detection_fn: 100%|██████████| 1000/1000 [00:01<00:00, 750.93it/s]

[{'visit_id': '100777', 'patient_id': '793', 'conditions': [['244', '131', '159', '153', '653', '79', '59', '2615']], 'procedures': [['216', '37']], 'drugs': [['J01MA02', 'V06DC01', 'N01AH01', 'A02BA03', 'N04BA02', 'A12BA01', 'N01AX10', 'N06AX05', 'N02BE01', 'A02BC03', 'A04AD01', 'N05CM05', 'N02AA01']], 'label': 0, 'gender': 'F', 'vitals': "{'SUBJECT_ID': 793.0, 'HADM_ID': 100777.0, 'diabp': 55.09782608695652, 'heartrate': 78.70526315789473, 'meanbp': 77.76812171936035, 'resprate': 17.745098039215687, 'spo2': 98.34285714285714, 'sysbp': 123.1086956521739, 'temperature': 68.55085570995624}"}]
[{'visit_id': '112982', 'patient_id': '794', 'conditions': [['96', '108', '55', '49', '663', '211']], 'procedures': [['43', '50']], 'drugs': [[]], 'label': 0, 'gender': 'F', 'vitals': "{'SUBJECT_ID': 794.0, 'HADM_ID': 112982.0, 'diabp': 33.63978494623656, 'heartrate': 89.53846153846153, 'meanbp': 67.8178701105806, 'resprate': 17.365591397849464, 'spo2': 97.8314606741573, 'sysbp': 76.84946236559139,

In [66]:

# ==========================================
# Step 3: Define the RNN Model
# ==========================================

from pyhealth.models import RNN
from pyhealth.trainer import Trainer

# Define the RNN model (LSTM-based)
model = RNN(
    dataset=train_dataset,
    feature_keys=["conditions", "procedures", "labs", "prescriptions"],
    label_key="label",
    mode="binary",
    embedding_dim=128,
    hidden_dim=64,
    num_layers=3,
    rnn_type="LSTM"
)

KeyboardInterrupt: 

In [ ]:
# ==========================================
# Step 4: Train and Evaluate the Model
# ==========================================

trainer = Trainer(model=model, device="cuda:0")
trainer.train(
    train_dataloader=train_dataset.get_dataloader(batch_size=64, shuffle=True),
    epochs=10,
    val_dataloader=val_dataset.get_dataloader(batch_size=64, shuffle=False),
    monitor="pr_auc",
)

# Evaluate on test set
trainer.evaluate(test_dataset.get_dataloader(batch_size=64))

In [ ]:
# ==========================================
# Step 5: Extract Model Activations for TCAV
# ==========================================

import torch

def extract_activations(model, dataloader):
    activations, labels = [], []
    model.eval()
    with torch.no_grad():
        for data in dataloader:
            inputs = model.prepare_input(data, model.feature_keys)
            output, hidden = model.model(inputs)
            activations.append(hidden[-1].cpu().numpy())  # last LSTM layer hidden state
            labels.append(data["label"].numpy())
    return activations, labels

train_acts, train_labels = extract_activations(model, train_dataset.get_dataloader(batch_size=64))

In [ ]:


# ==========================================
# Step 6: Build Concept Activation Vectors (CAVs)
# ==========================================

from sklearn.linear_model import LogisticRegression
import numpy as np

# Example concept: "NSAID prescription"
def define_concept(dataset, concept_key="NSAID"):
    concept_labels = []
    for patient in dataset.samples:
        prescriptions = patient["prescriptions"]
        concept_labels.append(1 if concept_key in prescriptions else 0)
    return np.array(concept_labels)

concept_labels = define_concept(train_dataset, "NSAID")


# Train CAV classifier
cav_classifier = LogisticRegression(max_iter=1000)
cav_classifier.fit(np.concatenate(train_acts), concept_labels)

In [ ]:



# ==========================================
# Step 7: Evaluate TCAV
# ==========================================

# Calculate concept influence
coefs = cav_classifier.coef_[0]
concept_importance = np.mean(coefs)
print(f"Concept Importance (NSAIDs): {concept_importance}")

In [ ]:
# ==========================================
# Step 8: Local Explanation for Single Patient
# ==========================================

# Choose a single patient from test set
patient_data = test_dataset.samples[0]
patient_dataloader = test_dataset.get_dataloader(batch_size=1, shuffle=False)
patient_acts, _ = extract_activations(model, patient_dataloader)

# Compute concept alignment
alignment_score = np.dot(patient_acts[0], coefs) / (np.linalg.norm(patient_acts[0]) * np.linalg.norm(coefs))
print(f"Alignment score for concept (NSAIDs) for patient 0: {alignment_score}")


In [ ]:

# ==========================================
# Step 9: Next Steps and Extensions
# ==========================================

# - Experiment with more clinical concepts (infection, gender, etc.)
# - Automate and expand TCAV analysis
# - Visualization of alignment and concept sensitivity scores
